# 02_pipeline — config-driven orchestration template

Build, check, publish, and record evidence for governed Fabric data pipelines.

This notebook is intentionally thin and copy-and-edit friendly. Users clone only DataFrame/config blocks for each source or target. FabricOps then orchestrates profiling, schema validation, stability enforcement, DQ enforcement, catalogue evidence, lineage, and runtime summary from those configs.

Flow:

1. Run `00_env_config`.
2. Import required functions.
3. Select the data agreement and capture run context.
4. Define source DataFrame/config blocks.
5. Collect source configs in `SOURCE_TABLES`.
6. Run source guardrails before transformation.
7. Transform source DataFrames into target DataFrames.
8. Define target DataFrame/config blocks.
9. Collect target configs in `TARGET_TABLES`.
10. Run target guardrails before writes.
11. Write targets only after all target guardrails pass.
12. Capture lineage and runtime summary evidence.

Schema, stability, and DQ remain separate guardrail concepts. The reusable orchestration helper only removes repeated notebook code.

## 1. Run `00_env_config`

Load the shared FabricOps environment, path configuration, sample metadata, and metadata lakehouse routing.

In [ ]:
%run 00_env_config


## 2. Import required functions

The notebook imports existing FabricOps callables directly for reads, profiling, guardrails, writes, lineage, and runtime-summary evidence.

In [ ]:
from datetime import datetime, timezone

from pyspark.sql import functions as F

from fabricops_kit import (
    enforce_catalogue_stability,
    enforce_dq_rules,
    get_selected_agreement,
    profile_dataframe,
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_warehouse_table,
    stop_if_failed,
    validate_schema,
    widget_select_agreement,
    write_catalogue_evidence,
    write_lakehouse_table,
    write_pipeline_lineage,
    write_pipeline_run_summary,
    write_warehouse_table,
)


## 3. Select data agreement and capture run context

Select the agreement that this pipeline satisfies. The selector registers this notebook in `METADATA_NOTEBOOK_REGISTRY` using the metadata target configured by `00_env_config`. The run context values are reused by guardrail evidence, lineage, and runtime summary writes.

In [ ]:
PIPELINE_STARTED_AT = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
RUN_ID = RUN_CONTEXT.run_id
ENV_NAME = ENV
PIPELINE_NAME = "CHANGE_ME_pipeline"

widget_select_agreement(
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    notebook_type="02_pipeline",
    pipeline_name=PIPELINE_NAME,
)

AGREEMENT = get_selected_agreement()
AGREEMENT_ID = AGREEMENT.get("agreement_id", "")
AGREEMENT_CONTRACT_VERSION = AGREEMENT.get("agreement_contract_version", AGREEMENT.get("contract_version", ""))
NOTEBOOK_REGISTRY_ID = AGREEMENT.get("notebook_registry_id", AGREEMENT.get("registration_id", ""))
NOTEBOOK_ID = AGREEMENT.get("notebook_id", RUN_CONTEXT.runtime_metadata.get("currentNotebookId", ""))


## 4. Source DataFrame/config blocks

FabricOps starts after a source DataFrame exists. Read each source with normal Spark code or FabricOps read helpers, then keep its guardrail settings in a small config block.

Each source block starts with a small editable parameter header. To add another source, copy the Source 01 block, replace `SOURCE_01`/`df_source_01` with `SOURCE_02`/`df_source_02`, edit the header values, and add the config to `SOURCE_TABLES`. Do not copy profiling, schema, stability, DQ, or catalogue-evidence code.

In [ ]:
DATASET_NAME = "CHANGE_ME_dataset"

# Source 01: clone this whole block for another source, then replace SOURCE_01
# with SOURCE_02 and edit only this parameter header plus table-specific guardrails.
SOURCE_01_KEY = "source_01"
SOURCE_01_DATASET_NAME = DATASET_NAME
SOURCE_01_LAYER = "source"
SOURCE_01_TABLE_NAME = "CHANGE_ME_source_table"
SOURCE_01_STAGE = "source"

df_source_01 = read_lakehouse_table(
    CONFIG,
    ENV_NAME,
    SOURCE_01_LAYER,
    SOURCE_01_TABLE_NAME,
    spark_session=spark,
)

SOURCE_01_CONFIG = {
    "key": SOURCE_01_KEY,
    "df": df_source_01,
    "dataset_name": SOURCE_01_DATASET_NAME,
    "table_name": SOURCE_01_TABLE_NAME,
    "stage": SOURCE_01_STAGE,
    "schema_preset": "allow_new_columns",
    "data_behavior": "changing",
    "stability_check_type": "watermark_slice_hash",
    "watermark_column": "CHANGE_ME_business_date",
    "watermark_value": None,
    "dq_preset": "approved_rules",
    "expected_schema": {
        "customer_id": "bigint",
        "event_ts": "string",
        "status": "string",
        "amount": "double",
        "email": "string",
        "country_code": "string",
    },
    "distribution_columns": ["status", "amount", "country_code"],
    "exclude_columns": None,
}

# Source 02 example: copy Source 01, replace SOURCE_01 with SOURCE_02,
# edit the parameter header, then add SOURCE_02_CONFIG to SOURCE_TABLES.
# SOURCE_02_KEY = "source_02"
# SOURCE_02_DATASET_NAME = DATASET_NAME
# SOURCE_02_LAYER = "source"
# SOURCE_02_TABLE_NAME = "CHANGE_ME_second_source_table"
# SOURCE_02_STAGE = "source"
#
# df_source_02 = read_lakehouse_csv(
#     CONFIG,
#     ENV_NAME,
#     SOURCE_02_LAYER,
#     "Files/CHANGE_ME/source_file.csv",
#     spark_session=spark,
#     header=True,
# )
#
# SOURCE_02_CONFIG = {
#     "key": SOURCE_02_KEY,
#     "df": df_source_02,
#     "dataset_name": SOURCE_02_DATASET_NAME,
#     "table_name": SOURCE_02_TABLE_NAME,
#     "stage": SOURCE_02_STAGE,
#     "schema_preset": "strict",
#     "data_behavior": "fixed",
#     "stability_check_type": "full_profile_hash",
#     "watermark_column": None,
#     "watermark_value": None,
#     "dq_preset": "skip",
#     "expected_schema": {"id": "bigint"},
#     "distribution_columns": [],
#     "exclude_columns": None,
# }

# Other common source read options:
# df_source_01 = read_warehouse_table(CONFIG, ENV_NAME, SOURCE_01_LAYER, "dbo", SOURCE_01_TABLE_NAME, spark_session=spark)
# df_source_01 = read_lakehouse_parquet(CONFIG, ENV_NAME, SOURCE_01_LAYER, "Files/CHANGE_ME/source_file.parquet", spark_session=spark)
# df_source_01 = read_lakehouse_excel(CONFIG, ENV_NAME, SOURCE_01_LAYER, "Files/CHANGE_ME/source_file.xlsx", spark_session=spark)
# df_source_01 = spark.read.table(SOURCE_01_TABLE_NAME)


### Optional: inspect a source schema

While authoring the notebook, inspect the current DataFrame schema and manually copy the reviewed expectation into `expected_schema`. Schema guardrails remain separate from DQ rules.

In [ ]:
# Optional authoring check: inspect the Spark schema before writing expected_schema.
df_source_01.printSchema()


## 5. Collect source configs

`SOURCE_TABLES` is the only list to update after cloning a source DataFrame/config block. FabricOps uses this list to run profiling, schema validation, stability checks, DQ checks, and catalogue evidence for every configured source.

In [ ]:
SOURCE_TABLES = [SOURCE_01_CONFIG]
# SOURCE_TABLES = [SOURCE_01_CONFIG, SOURCE_02_CONFIG]


## 6. Define reusable guardrail orchestration helpers

The helper loops through table configs and keeps schema, stability, and DQ outputs separate per table. It collects all guardrail results before stopping, so one blocking source or target does not hide results for later configured tables. Technical exceptions still stop immediately.

In [ ]:
def _table_key(table_config):
    return table_config["key"]


def _table_name(table_config):
    return table_config.get("table_name") or table_config.get("target_name") or table_config["key"]


def _guardrail_can_continue(result):
    return bool((result or {}).get("can_continue", True))


def build_guardrail_evidence_definitions(table_configs):
    definitions = {}
    for table_config in table_configs:
        table_key = _table_key(table_config)
        definition = {key: value for key, value in table_config.items() if key != "df"}
        definition["table_name"] = _table_name(table_config)
        definition["stage"] = table_config.get("stage", "target")
        if definition["stage"] == "target":
            definition["layer"] = table_config.get("target_layer", "unified")
            definition["kind"] = table_config.get("target_kind", "lakehouse")
            definition["mode"] = table_config.get("write_mode", "overwrite")
        definitions[table_key] = definition
    return definitions


def run_table_guardrails(
    table_configs,
    *,
    config,
    env,
    run_id,
    spark_session,
    agreement_id,
    agreement_contract_version,
    notebook_registry_id,
    notebook_id,
    pipeline_name,
):
    profiles = {}
    schema_results = {}
    stability_results = {}
    dq_results = {}
    failed_tables = []
    evidence_definitions = build_guardrail_evidence_definitions(table_configs)

    for table_config in table_configs:
        table_key = _table_key(table_config)
        table_name = _table_name(table_config)
        dataset_name = table_config.get("dataset_name", table_name)
        stage = table_config.get("stage", "target")
        dataframe = table_config["df"]

        profiles[table_key] = profile_dataframe(
            dataframe,
            table_name=table_name,
            # profile_dataframe automatically excludes FabricOps/DQ technical annotation columns
            # and unions those defaults with any table-specific exclude_columns.
            exclude_columns=table_config.get("exclude_columns"),
            include_distributions=True,
            distribution_columns=table_config.get("distribution_columns"),
        )

        schema_results[table_key] = validate_schema(
            dataframe,
            table_config["expected_schema"],
            preset=table_config.get("schema_preset", "strict"),
        )

        stability_results[table_key] = enforce_catalogue_stability(
            spark_session,
            dataframe,
            "METADATA_DATA_CATALOGUE",
            dataset_name,
            table_name,
            stage=stage,
            run_id=run_id,
            data_behavior=table_config.get("data_behavior", "changing"),
            stability_check_type=table_config.get("stability_check_type", "watermark_slice_hash"),
            watermark_column=table_config.get("watermark_column"),
            watermark_value=table_config.get("watermark_value"),
            exclude_columns=table_config.get("exclude_columns"),
            exclude_run_id=run_id,
            config=config,
            env=env,
        )

        if table_config.get("dq_preset", "approved_rules") == "skip":
            dq_results[table_key] = {
                "status": "skipped",
                "can_continue": True,
                "checks": [],
                "message": "DQ guardrail skipped by preset.",
            }
        else:
            dq_results[table_key] = enforce_dq_rules(
                dataframe,
                config,
                env,
                dataset_name,
                table_name,
                spark_session=spark_session,
            )

        if "dataframe" in dq_results[table_key]:
            table_config["df"] = dq_results[table_key]["dataframe"]

        table_can_continue = all(
            _guardrail_can_continue(result)
            for result in (schema_results[table_key], stability_results[table_key], dq_results[table_key])
        )
        if not table_can_continue:
            failed_tables.append(table_key)

    catalogue_status = write_catalogue_evidence(
        profiles,
        evidence_definitions,
        config=config,
        env=env,
        run_id=run_id,
        agreement_id=agreement_id,
        agreement_contract_version=agreement_contract_version,
        notebook_registry_id=notebook_registry_id,
        notebook_id=notebook_id,
        pipeline_name=pipeline_name,
        schema_results=schema_results,
        stability_results=stability_results,
        dq_results=dq_results,
    )

    return {
        "profiles": profiles,
        "schema_results": schema_results,
        "stability_results": stability_results,
        "dq_results": dq_results,
        "catalogue_status": catalogue_status,
        "evidence_definitions": evidence_definitions,
        "can_continue": not failed_tables,
        "failed_tables": failed_tables,
    }


def stop_if_any_guardrail_failed(guardrail_results):
    if guardrail_results.get("can_continue", True):
        return

    failed_tables = guardrail_results.get("failed_tables", [])
    stop_if_failed(
        {
            "status": "failed",
            "can_continue": False,
            "message": "Blocking guardrail failure for table(s): " + ", ".join(failed_tables),
            "failed_tables": failed_tables,
        }
    )


## 7. Run source guardrails before transformation

Source profiling, schema validation, stability checks, DQ checks, and catalogue evidence run for every config in `SOURCE_TABLES`. Results remain separated by guardrail type and table key. Transformation starts only after `stop_if_any_guardrail_failed(source_guardrail_results)` passes.

In [ ]:
source_dq_results = {}
source_guardrail_results = run_table_guardrails(
    SOURCE_TABLES,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    spark_session=spark,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
)

display(
    {
        "schema_results": source_guardrail_results["schema_results"],
        "stability_results": source_guardrail_results["stability_results"],
        "dq_results": source_guardrail_results["dq_results"],
        "catalogue_status": source_guardrail_results["catalogue_status"],
        "failed_tables": source_guardrail_results["failed_tables"],
    }
)
stop_if_any_guardrail_failed(source_guardrail_results)

source_profiles = source_guardrail_results["profiles"]
source_schema_results = source_guardrail_results["schema_results"]
source_stability_results = source_guardrail_results["stability_results"]
source_dq_results = source_guardrail_results["dq_results"]
source_catalogue_status = source_guardrail_results["catalogue_status"]
source_evidence_definitions = source_guardrail_results["evidence_definitions"]


## 8. Transform to target DataFrames

This is the main DIY section. Keep business logic here and reusable governance plumbing in the guardrail helper. Source guardrails have already passed, so transformations can safely depend on configured source DataFrames.

In [ ]:
df_target_01 = (
    SOURCE_01_CONFIG["df"]
    .withColumn(
        "amount_band",
        F.when(F.col("amount") >= F.lit(100), F.lit("high"))
        .when(F.col("amount") >= F.lit(25), F.lit("medium"))
        .otherwise(F.lit("low")),
    )
)

# Add more transformations or joins here. For many sources, use the config
# variables above or find a source by key from SOURCE_TABLES.


## 9. Target DataFrame/config blocks

Register each target DataFrame with write settings and the guardrails FabricOps should apply before publication.

Each target block starts with a small editable parameter header. To add another target, copy the Target 01 block, replace `TARGET_01`/`df_target_01` with `TARGET_02`/`df_target_02`, edit the header values, and add the config to `TARGET_TABLES`. Do not copy profiling, schema, stability, DQ, catalogue-evidence, or write orchestration code.

In [ ]:
# Target 01: clone this whole block for another target, then replace TARGET_01
# with TARGET_02 and edit only this parameter header plus table-specific guardrails.
TARGET_01_KEY = "target_01"
TARGET_01_DATASET_NAME = DATASET_NAME
TARGET_01_TABLE_NAME = "CHANGE_ME_target_table"
TARGET_01_LAYER = "unified"
TARGET_01_KIND = "lakehouse"
TARGET_01_WRITE_MODE = "overwrite"

AUDIT_CREATED_AT = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
df_target_01 = (
    df_target_01
    .withColumn("_fabricops_run_id", F.lit(RUN_ID))
    .withColumn("_fabricops_pipeline_name", F.lit(PIPELINE_NAME))
    .withColumn("_fabricops_created_at", F.lit(AUDIT_CREATED_AT))
)

TARGET_01_CONFIG = {
    "key": TARGET_01_KEY,
    "df": df_target_01,
    "dataset_name": TARGET_01_DATASET_NAME,
    "target_name": TARGET_01_TABLE_NAME,
    "target_layer": TARGET_01_LAYER,
    "target_kind": TARGET_01_KIND,
    "write_mode": TARGET_01_WRITE_MODE,
    "schema_preset": "strict",
    "data_behavior": "changing",
    "stability_check_type": "watermark_slice_hash",
    "watermark_column": "CHANGE_ME_business_date",
    "watermark_value": None,
    "dq_preset": "approved_rules",
    "expected_schema": {
        "customer_id": "bigint",
        "event_ts": "string",
        "status": "string",
        "amount": "double",
        "email": "string",
        "country_code": "string",
        "amount_band": "string",
        "_fabricops_run_id": "string",
        "_fabricops_pipeline_name": "string",
        "_fabricops_created_at": "string",
    },
    "distribution_columns": ["status", "amount", "amount_band", "country_code"],
    "partition_by": None,
    "repartition_by": None,
    "overwrite_schema": True,
}

# Target 02 example: create df_target_02 in the transform section, copy Target 01,
# replace TARGET_01 with TARGET_02, edit the parameter header, then add
# TARGET_02_CONFIG to TARGET_TABLES.
# TARGET_02_KEY = "target_02"
# TARGET_02_DATASET_NAME = DATASET_NAME
# TARGET_02_TABLE_NAME = "CHANGE_ME_second_target_table"
# TARGET_02_LAYER = "product"
# TARGET_02_KIND = "lakehouse"
# TARGET_02_WRITE_MODE = "overwrite"
#
# TARGET_02_CONFIG = {
#     "key": TARGET_02_KEY,
#     "df": df_target_02,
#     "dataset_name": TARGET_02_DATASET_NAME,
#     "target_name": TARGET_02_TABLE_NAME,
#     "target_layer": TARGET_02_LAYER,
#     "target_kind": TARGET_02_KIND,
#     "write_mode": TARGET_02_WRITE_MODE,
#     "schema_preset": "strict",
#     "data_behavior": "changing",
#     "stability_check_type": "watermark_slice_hash",
#     "watermark_column": "CHANGE_ME_business_date",
#     "watermark_value": None,
#     "dq_preset": "approved_rules",
#     "expected_schema": {"id": "bigint"},
#     "distribution_columns": [],
#     "partition_by": None,
#     "repartition_by": None,
#     "overwrite_schema": True,
# }


## 10. Collect target configs

`TARGET_TABLES` is the only list to update after cloning a target DataFrame/config block. FabricOps uses this list to run target guardrails and, after all target guardrails pass, write every configured target.

In [ ]:
TARGET_TABLES = [TARGET_01_CONFIG]
# TARGET_TABLES = [TARGET_01_CONFIG, TARGET_02_CONFIG]


## 11. Run target guardrails before writes

Target profiling, schema validation, stability checks, DQ checks, and catalogue evidence run for every config in `TARGET_TABLES`. Target writes do not happen unless `stop_if_any_guardrail_failed(target_guardrail_results)` passes.

Warning severity writes full data; error severity stops before write. No row filtering in v1.

In [ ]:
target_dq_results = {}
target_dfs = {_table_key(target_config): target_config["df"] for target_config in TARGET_TABLES}
target_guardrail_results = run_table_guardrails(
    TARGET_TABLES,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    spark_session=spark,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
)

display(
    {
        "schema_results": target_guardrail_results["schema_results"],
        "stability_results": target_guardrail_results["stability_results"],
        "dq_results": target_guardrail_results["dq_results"],
        "catalogue_status": target_guardrail_results["catalogue_status"],
        "failed_tables": target_guardrail_results["failed_tables"],
    }
)
target_profiles = target_guardrail_results["profiles"]
target_schema_results = target_guardrail_results["schema_results"]
target_stability_results = target_guardrail_results["stability_results"]
target_dq_results = target_guardrail_results["dq_results"]
for target_name in target_dq_results:
    print(target_dq_results[target_name])
    stop_if_failed(target_dq_results[target_name])
    if "dataframe" in target_dq_results[target_name]:
        target_dfs[target_name] = target_dq_results[target_name]["dataframe"]
target_catalogue_status = target_guardrail_results["catalogue_status"]
target_evidence_definitions = target_guardrail_results["evidence_definitions"]

stop_if_any_guardrail_failed(target_guardrail_results)


## 12. Write target tables

Only after all configured target guardrails pass, FabricOps loops through `TARGET_TABLES` and writes the target DataFrames. A blocking schema, stability, or DQ failure prevents every target write.

In [ ]:
target_write_status = {}
for target_config in TARGET_TABLES:
    target_key = target_config["key"]
    target_df = target_config["df"]
    target_kind = target_config.get("target_kind", "lakehouse")
    target_layer = target_config.get("target_layer", "unified")
    target_table = target_config.get("target_name", target_key)
    target_mode = target_config.get("write_mode", "overwrite")

    if target_kind == "lakehouse":
        write_lakehouse_table(
            target_df,
            CONFIG,
            ENV_NAME,
            target_layer,
            target_table,
            mode=target_mode,
            partition_by=target_config.get("partition_by"),
            repartition_by=target_config.get("repartition_by"),
            overwrite_schema=target_config.get("overwrite_schema", target_mode == "overwrite"),
        )
    elif target_kind == "warehouse":
        write_warehouse_table(
            target_df,
            CONFIG,
            ENV_NAME,
            target_layer,
            target_config.get("schema", "dbo"),
            target_table,
            mode=target_mode,
        )
    else:
        raise ValueError(f"Unsupported target kind for {target_key}: {target_kind}")
    target_write_status[target_key] = "written"


## 13. Capture many-to-many lineage

Define source-to-target relationships at the business level. FabricOps builds and writes metadata rows tied to the selected notebook registration.

In [ ]:
LINEAGE_RELATIONSHIPS = [
    {
        "sources": [SOURCE_01_KEY],
        "targets": [TARGET_01_KEY],
        "operation": f"derive amount band and publish {TARGET_01_TABLE_NAME}",
        "description": f"{SOURCE_01_TABLE_NAME} rows are transformed into {TARGET_01_TABLE_NAME}.",
    },
]

lineage_result = write_pipeline_lineage(
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    source_definitions=source_evidence_definitions,
    target_definitions=target_evidence_definitions,
    relationships=LINEAGE_RELATIONSHIPS,
    dataset_name=DATASET_NAME,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
)


## 14. Write runtime summary

Runtime evidence is stored in `METADATA_PIPELINE_RUNS` and displayed for operational support.

In [ ]:
catalogue_status = "written" if source_catalogue_status and target_catalogue_status else "not_written"

run_summary = write_pipeline_run_summary(
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
    started_at=PIPELINE_STARTED_AT,
    completed_at=datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    status="completed",
    source_definitions=source_evidence_definitions,
    target_definitions=target_evidence_definitions,
    source_schema_results=source_schema_results,
    target_schema_results=target_schema_results,
    source_stability_results=source_stability_results,
    target_stability_results=target_stability_results,
    source_dq_results=source_dq_results,
    target_dq_results=target_dq_results,
    lineage_status=lineage_result.get("status", "unknown"),
    catalogue_status=catalogue_status,
    message="Pipeline completed and metadata evidence was written.",
)

display(run_summary)
